<a href="https://colab.research.google.com/github/m-zayed5722/Miscellaneous-Projects/blob/main/GenAI_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GenAI Platform Lite — Unified Multi-Agent System (Colab)

This project integrates multiple GenAI components into a single pipeline:

User Query
→ Guardrails (PII + safety)
→ Router (decides tool)
→ Tool execution (SQL / RAG / Generic)
→ Response generation
→ Optional evaluation layer

Modules:
- GuardrailX (safety + PII)
- AgentRouter (intent + routing)
- SQL Engine (structured data)
- RAG Engine (document QA)
- Evaluation signals (quality tracking)

Goal:
Simulate a production-grade GenAI orchestration system.

In [1]:
!pip -q install duckdb pandas rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 55.9 MB/s eta 0:00:00


In [2]:
import re
import duckdb
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Any, List
from rapidfuzz import fuzz

In [3]:
df_apps = pd.DataFrame([
    {"app": "Atlas", "team": "Platform", "cost": 12000, "incidents": 2},
    {"app": "Nimbus", "team": "Payments", "cost": 22000, "incidents": 5},
    {"app": "Orion", "team": "Core", "cost": 18000, "incidents": 1},
    {"app": "Pulse", "team": "Mobile", "cost": 14000, "incidents": 4},
])

con = duckdb.connect()
con.execute("CREATE TABLE apps AS SELECT * FROM df_apps")

In [4]:
DOCS = [
    {"title": "RAG", "text": "RAG combines retrieval with generation to ground LLM outputs."},
    {"title": "Guardrails", "text": "Guardrails include PII removal, policy checks, and safe completion rules."},
    {"title": "Routing", "text": "Agent routing selects tools based on intent classification."},
]

In [5]:
PII_REGEX = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b|\b\d{3}[-]?\d{3}[-]?\d{4}\b")

def guardrails(text: str):
    pii_found = bool(PII_REGEX.search(text))
    risky = any(k in text.lower() for k in ["hack", "steal", "bypass", "exploit"])

    blocked = risky
    redacted = PII_REGEX.sub("[REDACTED]", text)

    return {
        "blocked": blocked,
        "pii_detected": pii_found,
        "safe_text": redacted
    }

In [6]:
def route_query(q: str):
    ql = q.lower()

    if any(k in ql for k in ["cost", "team", "incidents", "apps"]):
        return "sql"
    if any(k in ql for k in ["document", "rag", "explain", "notes"]):
        return "rag"
    return "generic"

In [7]:
def sql_tool(q: str):
    if "cost" in q.lower():
        sql = "SELECT app, cost FROM apps ORDER BY cost DESC"
    elif "incidents" in q.lower():
        sql = "SELECT app, incidents FROM apps ORDER BY incidents DESC"
    else:
        sql = "SELECT * FROM apps"

    return {
        "tool": "sql",
        "sql": sql,
        "result": con.execute(sql).fetchdf()
    }

In [8]:
def rag_tool(q: str):
    best = max(DOCS, key=lambda d: fuzz.partial_ratio(q.lower(), d["text"].lower()))
    return {
        "tool": "rag",
        "result": best["text"],
        "source": best["title"]
    }

In [9]:
def generic_tool(q: str):
    return {
        "tool": "generic",
        "result": f"Processed query: {q}"
    }

In [10]:
def run_system(query: str):

    # 1. Guardrails
    g = guardrails(query)
    if g["blocked"]:
        return {
            "status": "blocked",
            "reason": "Unsafe query detected",
            "details": g
        }

    # 2. Routing
    tool = route_query(g["safe_text"])

    # 3. Execute tool
    if tool == "sql":
        out = sql_tool(g["safe_text"])
    elif tool == "rag":
        out = rag_tool(g["safe_text"])
    else:
        out = generic_tool(g["safe_text"])

    # 4. Response packaging
    return {
        "status": "ok",
        "route": tool,
        "guardrails": g,
        "output": out
    }

In [11]:
queries = [
    "Which app has highest cost?",
    "Explain RAG from documents",
    "My email is test@example.com, show incidents",
    "How do I bypass system security?"
]

for q in queries:
    print("\n" + "="*80)
    print("QUERY:", q)
    res = run_system(q)
    print(res)


QUERY: Which app has highest cost?
{'status': 'ok', 'route': 'sql', 'guardrails': {'blocked': False, 'pii_detected': False, 'safe_text': 'Which app has highest cost?'}, 'output': {'tool': 'sql', 'sql': 'SELECT app, cost FROM apps ORDER BY cost DESC', 'result':       app   cost
0  Nimbus  22000
1   Orion  18000
2   Pulse  14000
3   Atlas  12000}}

QUERY: Explain RAG from documents
{'status': 'ok', 'route': 'rag', 'guardrails': {'blocked': False, 'pii_detected': False, 'safe_text': 'Explain RAG from documents'}, 'output': {'tool': 'rag', 'result': 'RAG combines retrieval with generation to ground LLM outputs.', 'source': 'RAG'}}

QUERY: My email is test@example.com, show incidents
{'status': 'ok', 'route': 'sql', 'guardrails': {'blocked': False, 'pii_detected': True, 'safe_text': 'My email is [REDACTED], show incidents'}, 'output': {'tool': 'sql', 'sql': 'SELECT app, incidents FROM apps ORDER BY incidents DESC', 'result':       app  incidents
0  Nimbus          5
1   Pulse          4
2 

In [12]:
def score_output(result):
    if result["status"] != "ok":
        return 0
    base = 1

    if result["route"] == "sql":
        base += 1
    if result["route"] == "rag":
        base += 1

    if result["guardrails"]["pii_detected"]:
        base -= 0.5

    return base

for q in queries:
    res = run_system(q)
    print(q, "→ score:", score_output(res))

Which app has highest cost? → score: 2
Explain RAG from documents → score: 2
My email is test@example.com, show incidents → score: 1.5
How do I bypass system security? → score: 0
